# 🚀 State-of-the-Art Deepfake Detection Model Training

## Complete Production-Ready Training Pipeline

This notebook trains a **state-of-the-art** deepfake detection model optimized for:
- **Kaggle GPU** (T4/P100/V100)
- **140k Real and Fake Faces Dataset**
- **Backend Integration** (saves in correct format)

### Features:
- ✅ **ConvNeXt Large** (SOTA CNN architecture)
- ✅ **Advanced Augmentation** (MixUp, CutMix, Multi-scale)
- ✅ **Focal Loss** for class imbalance
- ✅ **MTCNN Face Detection** (high quality)
- ✅ **Proper Model Saving** (compatible with your backend)
- ✅ **Comprehensive Evaluation** (AUC, F1, Precision, Recall)

**Expected Performance**: 94-97% Accuracy, 0.97-0.99 AUC-ROC


## 📦 Step 1: Install Dependencies & Setup


In [ ]:
# Install required packages
!pip install -q timm facenet-pytorch scikit-learn tqdm

# Fix PIL compatibility issue - use version compatible with facenet-pytorch
# facenet-pytorch requires Pillow<10.3.0,>=10.2.0
# We need >=10.0.0 for torchvision compatibility
# Uninstall existing Pillow and install correct version
!pip uninstall -y Pillow
!pip install "Pillow>=10.2.0,<10.3.0"

import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import numpy as np
from PIL import Image
import cv2
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, 
    roc_auc_score, confusion_matrix, classification_report
)
from tqdm import tqdm
from pathlib import Path
import json
import random
import warnings
warnings.filterwarnings('ignore')

import timm
from facenet_pytorch import MTCNN

print("✅ All packages imported successfully")
print(f"✅ PyTorch version: {torch.__version__}")
print(f"✅ CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"✅ GPU: {torch.cuda.get_device_name(0)}")
    print(f"✅ GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")


## ⚙️ Step 2: Configuration


In [ ]:
# ============================================================================
# CONFIGURATION - STATE-OF-THE-ART SETTINGS + OOM OPTIMIZATIONS
# ============================================================================

CONFIG = {
    # Model Architecture
    'model_name': 'convnext_large',  # SOTA: convnext_large, efficientnet_b4, vit_large_patch16_224
    'num_classes': 2,  # Real (0) vs Fake (1)
    
    # Training Hyperparameters - T4 GPU OPTIMIZED (16GB)
    'image_size': 224,
    'batch_size': 8,  # T4 OPTIMIZED: Reduced to 8 for ConvNeXt Large
    'gradient_accumulation_steps': 4,  # Effective batch = 8 * 4 = 32
    'num_epochs': 50,
    'learning_rate': 1e-4,
    'weight_decay': 1e-4,
    'num_workers': 0,  # Set to 0 to fix worker process issues with face detection
    'device': 'cuda' if torch.cuda.is_available() else 'cpu',
    
    # CUDA OOM OPTIMIZATIONS - AGGRESSIVE FOR T4
    'use_mixed_precision': True,  # Mixed precision training (saves ~50% memory)
    'use_gradient_checkpointing': False,  # Enable if still OOM (trades compute for memory)
    'empty_cache_frequency': 5,  # Clear cache every 5 batches (more frequent)
    'max_memory_allocated_gb': 14.0,  # Safety limit for T4 (16GB - 2GB buffer)
    
    # Dataset Paths (Kaggle)
    'train_path': '/kaggle/input/140k-real-and-fake-faces/real_vs_fake/real-vs-fake/train',
    'valid_path': '/kaggle/input/140k-real-and-fake-faces/real_vs_fake/real-vs-fake/valid',
    'test_path': '/kaggle/input/140k-real-and-fake-faces/real_vs_fake/real-vs-fake/test',
    
    # Model Saving
    'save_path': '/kaggle/working/best_deepfake_model.pth',
    'final_model_path': '/kaggle/working/deepfake_detection_sota.pth',
    
    # Advanced Training Techniques - T4 OPTIMIZED
    'use_face_crop': True,  # Face detection and cropping
    'use_mixup': False,  # DISABLED for T4 (saves memory, can enable if no OOM)
    'mixup_alpha': 0.4,
    'use_cutmix': False,  # DISABLED for T4 (saves memory, can enable if no OOM)
    'cutmix_alpha': 1.0,
    'label_smoothing': 0.1,
    'use_focal_loss': True,
    'focal_alpha': 0.25,
    'focal_gamma': 2.0,
    'use_multi_scale': False,  # DISABLED for memory efficiency
    'multi_scale_sizes': [224, 256, 288],
}

print("="*70)
print("📋 CONFIGURATION")
print("="*70)
print(f"✅ Device: {CONFIG['device']}")
print(f"✅ Model: {CONFIG['model_name']} (SOTA Architecture)")
print(f"✅ Batch Size: {CONFIG['batch_size']} (Effective: {CONFIG['batch_size'] * CONFIG['gradient_accumulation_steps']} with grad accumulation)")
print(f"✅ Epochs: {CONFIG['num_epochs']}")
print(f"✅ Learning Rate: {CONFIG['learning_rate']}")
print(f"✅ T4 GPU Optimizations (16GB):")
print(f"   - Mixed Precision: {CONFIG['use_mixed_precision']} (saves ~50% memory)")
print(f"   - Gradient Accumulation: {CONFIG['gradient_accumulation_steps']} steps")
print(f"   - Empty Cache Frequency: Every {CONFIG['empty_cache_frequency']} batches")
print(f"   - Memory Limit: {CONFIG['max_memory_allocated_gb']} GB")
print(f"   - Num Workers: {CONFIG['num_workers']} (0 = no multiprocessing, fixes worker issue)")
print(f"✅ Advanced Techniques:")
print(f"   - MixUp: {CONFIG['use_mixup']} (disabled for T4)")
print(f"   - CutMix: {CONFIG['use_cutmix']} (disabled for T4)")
print(f"   - Focal Loss: {CONFIG['use_focal_loss']}")
print(f"   - Multi-Scale: {CONFIG['use_multi_scale']} (disabled for memory)")
print(f"\n⚠️  T4 GPU Settings: Batch=8, GradAccum=4, MixUp/CutMix=OFF")
print(f"   If still OOM, reduce batch_size to 4 or use 'efficientnet_b4' model")
print("="*70)

# Memory check
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    memory_allocated = torch.cuda.memory_allocated(0) / 1e9
    memory_reserved = torch.cuda.memory_reserved(0) / 1e9
    memory_total = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"\n💾 GPU Memory Status:")
    print(f"   Total: {memory_total:.2f} GB")
    print(f"   Reserved: {memory_reserved:.2f} GB")
    print(f"   Allocated: {memory_allocated:.2f} GB")
    print(f"   Free: {memory_total - memory_reserved:.2f} GB")


## 🛡️ T4 GPU (16GB) Optimizations

This notebook is **specifically optimized for T4 GPUs** with aggressive memory savings:

1. **Mixed Precision Training** (AMP) - Saves ~50% GPU memory
2. **Gradient Accumulation** (4 steps) - Effective batch=32 with batch_size=8
3. **Automatic Cache Clearing** - Every 5 batches (very frequent)
4. **OOM Error Handling** - Automatic recovery and graceful degradation
5. **Memory Monitoring** - Real-time GPU memory usage with safety limits
6. **Optimized Data Loading** - Reduced pin_memory and workers
7. **MixUp/CutMix Disabled** - Saves memory (can enable if no OOM)

**Current T4 Settings:**
- `batch_size=8` (reduced from 16)
- `gradient_accumulation_steps=4` (effective batch=32)
- `use_mixup=False` (disabled)
- `use_cutmix=False` (disabled)
- `empty_cache_frequency=5` (very frequent)

**If you STILL get OOM errors:**
1. **Reduce batch_size to 4**: Change `'batch_size': 4` and `'gradient_accumulation_steps': 8`
2. **Use smaller model**: Change `'model_name': 'efficientnet_b4'` (much smaller, still excellent)
3. **Enable gradient checkpointing**: Set `'use_gradient_checkpointing': True` (slower but saves memory)


## 🔍 Step 3: Face Detection Setup


In [ ]:
# Initialize MTCNN face detector (best quality)
print("🔍 Initializing Face Detector...")
mtcnn = MTCNN(image_size=224, margin=40, device=CONFIG['device'])
print("✅ MTCNN face detector loaded")

# OpenCV fallback
face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')

def detect_and_crop_face_advanced(image):
    """Advanced face detection with MTCNN or OpenCV fallback"""
    try:
        # MTCNN returns (image, prob, landmarks)
        face_img, prob = mtcnn(image, return_prob=True)
        if face_img is not None and prob > 0.9:
            # Convert tensor to PIL
            face_array = face_img.permute(1, 2, 0).cpu().numpy()
            face_array = (face_array * 255).astype(np.uint8)
            face_array = np.clip(face_array, 0, 255)
            return Image.fromarray(face_array)
    except Exception as e:
        pass
    
    # OpenCV fallback
    try:
        img_array = np.array(image)
        if len(img_array.shape) == 2:
            img_array = cv2.cvtColor(img_array, cv2.COLOR_GRAY2RGB)
        
        gray = cv2.cvtColor(img_array, cv2.COLOR_RGB2GRAY)
        faces = face_cascade.detectMultiScale(gray, 1.3, 5)
        
        if len(faces) > 0:
            x, y, w, h = faces[0]
            # Add padding
            padding = int(w * 0.3)
            x1 = max(0, x - padding)
            y1 = max(0, y - padding)
            x2 = min(img_array.shape[1], x + w + padding)
            y2 = min(img_array.shape[0], y + h + padding)
            return image.crop((x1, y1, x2, y2))
    except:
        pass
    
    # Center crop fallback
    w, h = image.size
    size = min(w, h)
    left = (w - size) // 2
    top = (h - size) // 2
    return image.crop((left, top, left + size, top + size))

print("✅ Face detection function ready")


## 📊 Step 4: Advanced Augmentation Classes


In [ ]:
class MixUp:
    """MixUp augmentation for better generalization"""
    def __init__(self, alpha=0.4):
        self.alpha = alpha
    
    def __call__(self, batch):
        if random.random() > 0.5:
            return batch
        
        images, labels = batch
        batch_size = images.size(0)
        indices = torch.randperm(batch_size).to(images.device)
        
        lam = np.random.beta(self.alpha, self.alpha)
        mixed_images = lam * images + (1 - lam) * images[indices]
        y_a, y_b = labels, labels[indices]
        
        return mixed_images, y_a, y_b, lam

class CutMix:
    """CutMix augmentation"""
    def __init__(self, alpha=1.0):
        self.alpha = alpha
    
    def __call__(self, batch):
        if random.random() > 0.5:
            return batch
        
        images, labels = batch
        batch_size = images.size(0)
        indices = torch.randperm(batch_size).to(images.device)
        
        lam = np.random.beta(self.alpha, self.alpha)
        bbx1, bby1, bbx2, bby2 = self.rand_bbox(images.size(), lam)
        images[:, :, bbx1:bbx2, bby1:bby2] = images[indices, :, bbx1:bbx2, bby1:bby2]
        
        lam = 1 - ((bbx2 - bbx1) * (bby2 - bby1) / (images.size()[-1] * images.size()[-2]))
        y_a, y_b = labels, labels[indices]
        
        return images, y_a, y_b, lam
    
    def rand_bbox(self, size, lam):
        W, H = size[2], size[3]
        cut_rat = np.sqrt(1. - lam)
        cut_w, cut_h = np.int(W * cut_rat), np.int(H * cut_rat)
        cx, cy = np.random.randint(W), np.random.randint(H)
        bbx1 = np.clip(cx - cut_w // 2, 0, W)
        bby1 = np.clip(cy - cut_h // 2, 0, H)
        bbx2 = np.clip(cx + cut_w // 2, 0, W)
        bby2 = np.clip(cy + cut_h // 2, 0, H)
        return bbx1, bby1, bbx2, bby2

print("✅ Augmentation classes defined")


## 🎯 Step 5: Advanced Loss Functions


In [ ]:
class FocalLoss(nn.Module):
    """Focal Loss for handling class imbalance"""
    def __init__(self, alpha=0.25, gamma=2.0, reduction='mean'):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction
    
    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(inputs, targets, reduction='none')
        pt = torch.exp(-ce_loss)
        focal_loss = self.alpha * (1 - pt) ** self.gamma * ce_loss
        
        if self.reduction == 'mean':
            return focal_loss.mean()
        elif self.reduction == 'sum':
            return focal_loss.sum()
        return focal_loss

class LabelSmoothingCrossEntropy(nn.Module):
    """Label smoothing for better generalization"""
    def __init__(self, smoothing=0.1):
        super().__init__()
        self.smoothing = smoothing
    
    def forward(self, pred, target):
        log_prob = F.log_softmax(pred, dim=-1)
        nll_loss = -log_prob.gather(dim=-1, index=target.unsqueeze(1)).squeeze(1)
        smooth_loss = -log_prob.mean(dim=-1)
        loss = (1 - self.smoothing) * nll_loss + self.smoothing * smooth_loss
        return loss.mean()

print("✅ Loss functions defined")


## 📁 Step 6: Dataset Class


In [ ]:
class DeepfakeDataset(Dataset):
    """Dataset class for deepfake detection"""
    def __init__(self, data_dir, transform=None, use_face_crop=True, is_training=False):
        self.data_dir = Path(data_dir)
        self.transform = transform
        self.use_face_crop = use_face_crop
        self.is_training = is_training
        
        # Debug: Print the path being used
        print(f"\n🔍 Loading dataset from: {self.data_dir}")
        print(f"   Path exists: {self.data_dir.exists()}")
        
        real_dir = self.data_dir / 'real'
        fake_dir = self.data_dir / 'fake'
        
        print(f"   Real dir: {real_dir} (exists: {real_dir.exists()})")
        print(f"   Fake dir: {fake_dir} (exists: {fake_dir.exists()})")
        
        self.image_paths = []
        self.labels = []
        
        # Real images (label 0)
        if real_dir.exists():
            real_images = (list(real_dir.glob('*.jpg')) + 
                          list(real_dir.glob('*.png')) + 
                          list(real_dir.glob('*.jpeg')) + 
                          list(real_dir.glob('*.JPG')))
            self.image_paths.extend(real_images)
            self.labels.extend([0] * len(real_images))
            print(f"   ✅ Found {len(real_images):,} real images")
        else:
            print(f"   ⚠️ Real directory not found: {real_dir}")
        
        # Fake images (label 1)
        if fake_dir.exists():
            fake_images = (list(fake_dir.glob('*.jpg')) + 
                          list(fake_dir.glob('*.png')) + 
                          list(fake_dir.glob('*.jpeg')) + 
                          list(fake_dir.glob('*.JPG')))
            self.image_paths.extend(fake_images)
            self.labels.extend([1] * len(fake_images))
            print(f"   ✅ Found {len(fake_images):,} fake images")
        else:
            print(f"   ⚠️ Fake directory not found: {fake_dir}")
        
        if len(self.image_paths) == 0:
            raise ValueError(f"No images found in {data_dir}")
        
        print(f"   📂 Loaded {len(self.image_paths):,} images from {self.data_dir.name}")
        print(f"      Real: {sum(1 for l in self.labels if l == 0):,}")
        print(f"      Fake: {sum(1 for l in self.labels if l == 1):,}")
    
    def __len__(self):
        return len(self.image_paths)
    
    def _detect_and_crop_face(self, image):
        """Embedded face detection function - works with num_workers=0 or can access globals"""
        try:
            # Try to use global mtcnn if available (works with num_workers=0)
            if 'mtcnn' in globals():
                face_img, prob = mtcnn(image, return_prob=True)
                if face_img is not None and prob > 0.9:
                    face_array = face_img.permute(1, 2, 0).cpu().numpy()
                    face_array = (face_array * 255).astype(np.uint8)
                    face_array = np.clip(face_array, 0, 255)
                    return Image.fromarray(face_array)
        except:
            pass
        
        # OpenCV fallback
        try:
            img_array = np.array(image)
            if len(img_array.shape) == 2:
                img_array = cv2.cvtColor(img_array, cv2.COLOR_GRAY2RGB)
            
            gray = cv2.cvtColor(img_array, cv2.COLOR_RGB2GRAY)
            face_cascade_local = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')
            faces = face_cascade_local.detectMultiScale(gray, 1.3, 5)
            
            if len(faces) > 0:
                x, y, w, h = faces[0]
                padding = int(w * 0.3)
                x1 = max(0, x - padding)
                y1 = max(0, y - padding)
                x2 = min(img_array.shape[1], x + w + padding)
                y2 = min(img_array.shape[0], y + h + padding)
                return image.crop((x1, y1, x2, y2))
        except:
            pass
        
        # Center crop fallback
        w, h = image.size
        size = min(w, h)
        left = (w - size) // 2
        top = (h - size) // 2
        return image.crop((left, top, left + size, top + size))
    
    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        label = self.labels[idx]
        
        # Load image with error handling
        max_retries = 3
        for retry in range(max_retries):
            try:
                image = Image.open(img_path).convert('RGB')
                break
            except Exception as e:
                if retry == max_retries - 1:
                    image = Image.new('RGB', (224, 224), (0, 0, 0))
                else:
                    idx = (idx + 1) % len(self.image_paths)
                    img_path = self.image_paths[idx]
                    label = self.labels[idx]
        
        # Face detection and cropping - use embedded method
        if self.use_face_crop:
            image = self._detect_and_crop_face(image)
        
        # Multi-scale training
        if self.is_training and CONFIG['use_multi_scale']:
            size = random.choice(CONFIG['multi_scale_sizes'])
            temp_transform = transforms.Compose([
                transforms.Resize((size, size)),
                transforms.ToTensor(),
                transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
            ])
            image = temp_transform(image)
        else:
            if self.transform:
                image = self.transform(image)
            else:
                image = transforms.Compose([
                    transforms.Resize((224, 224)),
                    transforms.ToTensor(),
                    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
                ])(image)
        
        return image, label


In [ ]:
# Data transforms
train_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomCrop(224),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.1),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.1),
    transforms.RandomRotation(15),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1), scale=(0.9, 1.1)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    transforms.RandomErasing(p=0.2, scale=(0.02, 0.33)),
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Load datasets
print("\n📂 Loading Datasets...")
train_dataset = DeepfakeDataset(CONFIG['train_path'], transform=train_transform, use_face_crop=CONFIG['use_face_crop'], is_training=True)
valid_dataset = DeepfakeDataset(CONFIG['valid_path'], transform=val_transform, use_face_crop=CONFIG['use_face_crop'], is_training=False)
test_dataset = DeepfakeDataset(CONFIG['test_path'], transform=val_transform, use_face_crop=CONFIG['use_face_crop'], is_training=False)

# Data loaders - FIXED: num_workers=0 to avoid worker process issues with face detection
train_loader = DataLoader(
    train_dataset, 
    batch_size=CONFIG['batch_size'], 
    shuffle=True, 
    num_workers=0,  # Set to 0 to fix worker process issues
    pin_memory=False, 
    drop_last=True, 
    persistent_workers=False  # Must be False when num_workers=0
)

valid_loader = DataLoader(
    valid_dataset, 
    batch_size=CONFIG['batch_size'], 
    shuffle=False, 
    num_workers=0,  # Set to 0 to fix worker process issues
    pin_memory=False, 
    persistent_workers=False  # Must be False when num_workers=0
)

test_loader = DataLoader(
    test_dataset, 
    batch_size=CONFIG['batch_size'], 
    shuffle=False, 
    num_workers=0,  # Set to 0 to fix worker process issues
    pin_memory=False, 
    persistent_workers=False  # Must be False when num_workers=0
)

print(f"\n✅ Dataset Summary:")
print(f"   Train: {len(train_dataset):,} images")
print(f"   Valid: {len(valid_dataset):,} images")
print(f"   Test:  {len(test_dataset):,} images")


## 🏗️ Step 8: Create SOTA Model


In [ ]:
def create_sota_model(model_name='convnext_large', num_classes=2):
    """Create state-of-the-art model using timm"""
    try:
        model = timm.create_model(model_name, pretrained=True, num_classes=num_classes, drop_rate=0.3, drop_path_rate=0.2)
        print(f"✅ Created {model_name} with pretrained weights")
        return model
    except Exception as e:
        print(f"⚠️ Error creating {model_name}: {e}")
        print("Falling back to EfficientNet-B4")
        from torchvision.models import efficientnet_b4, EfficientNet_B4_Weights
        model = efficientnet_b4(weights=EfficientNet_B4_Weights.IMAGENET1K_V1)
        model.classifier = nn.Sequential(nn.Dropout(0.4), nn.Linear(model.classifier[1].in_features, num_classes))
        return model

print("\n🏗️  Creating Model...")
model = create_sota_model(CONFIG['model_name'], num_classes=CONFIG['num_classes'])
model = model.to(CONFIG['device'])

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"✅ Model: {total_params:,} total params, {trainable_params:,} trainable")


## 🎯 Step 9: Loss Function & Optimizer Setup


In [ ]:
# Loss function
if CONFIG['use_focal_loss']:
    criterion = FocalLoss(alpha=CONFIG['focal_alpha'], gamma=CONFIG['focal_gamma'])
    print("✅ Using Focal Loss")
elif CONFIG['label_smoothing'] > 0:
    criterion = LabelSmoothingCrossEntropy(smoothing=CONFIG['label_smoothing'])
    print("✅ Using Label Smoothing")
else:
    criterion = nn.CrossEntropyLoss()
    print("✅ Using Standard CrossEntropy Loss")

# Initialize augmentation
mixup = MixUp(alpha=CONFIG['mixup_alpha']) if CONFIG['use_mixup'] else None
cutmix = CutMix(alpha=CONFIG['cutmix_alpha']) if CONFIG['use_cutmix'] else None

# Optimizer
optimizer = optim.AdamW(model.parameters(), lr=CONFIG['learning_rate'], weight_decay=CONFIG['weight_decay'], betas=(0.9, 0.999))

# Cosine annealing with warm restarts
scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=10, T_mult=2, eta_min=1e-6)

# Mixed precision scaler (for OOM prevention)
scaler = torch.cuda.amp.GradScaler() if CONFIG['use_mixed_precision'] else None
if CONFIG['use_mixed_precision']:
    print("✅ Mixed precision training enabled (saves ~50% memory)")
else:
    print("⚠️ Mixed precision disabled")

print("✅ Optimizer and scheduler configured")


## 🚂 Step 10: Training Functions


In [ ]:
def train_epoch_advanced(model, loader, criterion, optimizer, device, epoch, scaler=None):
    """Advanced training epoch with MixUp/CutMix + OOM optimizations"""
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    # Clear cache at start
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    
    pbar = tqdm(loader, desc=f'Epoch {epoch+1} [Train]')
    optimizer.zero_grad()  # Zero grad at start for gradient accumulation
    
    for batch_idx, (images, labels) in enumerate(pbar):
        try:
            images, labels = images.to(device, non_blocking=True), labels.to(device, non_blocking=True)
            
            # Mixed precision forward pass
            with torch.cuda.amp.autocast(enabled=CONFIG['use_mixed_precision']):
                # Apply MixUp or CutMix
                if CONFIG['use_mixup'] and random.random() < 0.5:
                    mixed_images, y_a, y_b, lam = mixup((images, labels))
                    outputs = model(mixed_images)
                    loss = lam * criterion(outputs, y_a) + (1 - lam) * criterion(outputs, y_b)
                elif CONFIG['use_cutmix'] and random.random() < 0.5:
                    mixed_images, y_a, y_b, lam = cutmix((images, labels))
                    outputs = model(mixed_images)
                    loss = lam * criterion(outputs, y_a) + (1 - lam) * criterion(outputs, y_b)
                else:
                    outputs = model(images)
                    loss = criterion(outputs, labels)
                
                # Scale loss for gradient accumulation
                loss = loss / CONFIG['gradient_accumulation_steps']
            
            # Backward pass with mixed precision
            if scaler is not None:
                scaler.scale(loss).backward()
            else:
                loss.backward()
            
            # Gradient accumulation: only step every N batches
            if (batch_idx + 1) % CONFIG['gradient_accumulation_steps'] == 0:
                if scaler is not None:
                    scaler.unscale_(optimizer)
                    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                    scaler.step(optimizer)
                    scaler.update()
                else:
                    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                    optimizer.step()
                optimizer.zero_grad()
            
            # Update metrics (unscale loss for display)
            running_loss += loss.item() * CONFIG['gradient_accumulation_steps']
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
            
            # Memory management: clear cache periodically
            if (batch_idx + 1) % CONFIG['empty_cache_frequency'] == 0:
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()
            
            # Memory monitoring
            if torch.cuda.is_available() and batch_idx % 50 == 0:
                memory_used = torch.cuda.memory_allocated(0) / 1e9
                pbar.set_postfix({
                    'loss': f'{running_loss/(batch_idx+1):.4f}',
                    'acc': f'{100.*correct/total:.2f}%',
                    'lr': f'{optimizer.param_groups[0]["lr"]:.2e}',
                    'mem': f'{memory_used:.2f}GB'
                })
            else:
                pbar.set_postfix({
                    'loss': f'{running_loss/(batch_idx+1):.4f}',
                    'acc': f'{100.*correct/total:.2f}%',
                    'lr': f'{optimizer.param_groups[0]["lr"]:.2e}'
                })
        
        except RuntimeError as e:
            if "out of memory" in str(e):
                print(f"\n⚠️ OOM Error at batch {batch_idx}! Attempting recovery...")
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()
                # Try with smaller batch or skip
                if batch_idx == 0:
                    print("❌ OOM on first batch! Reduce batch_size in CONFIG.")
                    raise
                continue
            else:
                raise
    
    # Final gradient step if needed
    if (len(loader) % CONFIG['gradient_accumulation_steps']) != 0:
        if scaler is not None:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
        else:
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
        optimizer.zero_grad()
    
    return running_loss / len(loader), 100. * correct / total

def validate(model, loader, criterion, device):
    """Validation function with comprehensive metrics + OOM protection"""
    model.eval()
    running_loss = 0.0
    all_preds = []
    all_labels = []
    
    # Clear cache before validation
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    
    with torch.no_grad():
        for images, labels in tqdm(loader, desc='Validating'):
            try:
                images, labels = images.to(device, non_blocking=True), labels.to(device, non_blocking=True)
                
                # Mixed precision for validation too
                with torch.cuda.amp.autocast(enabled=CONFIG['use_mixed_precision']):
                    outputs = model(images)
                    loss = criterion(outputs, labels)
                
                running_loss += loss.item()
                probs = torch.softmax(outputs, dim=1)
                all_preds.extend(probs[:, 1].cpu().numpy())
                all_labels.extend(labels.cpu().numpy())
                
                # Clear cache periodically
                if len(all_preds) % 100 == 0 and torch.cuda.is_available():
                    torch.cuda.empty_cache()
            
            except RuntimeError as e:
                if "out of memory" in str(e):
                    print(f"\n⚠️ OOM during validation! Clearing cache...")
                    if torch.cuda.is_available():
                        torch.cuda.empty_cache()
                    continue
                else:
                    raise
    
    epoch_loss = running_loss / len(loader)
    all_preds = np.array(all_preds)
    all_labels = np.array(all_labels)
    all_preds_binary = (all_preds > 0.5).astype(int)
    
    acc = accuracy_score(all_labels, all_preds_binary)
    precision = precision_score(all_labels, all_preds_binary, zero_division=0)
    recall = recall_score(all_labels, all_preds_binary, zero_division=0)
    f1 = f1_score(all_labels, all_preds_binary, zero_division=0)
    auc = roc_auc_score(all_labels, all_preds)
    
    return epoch_loss, acc, precision, recall, f1, auc

print("✅ Training functions defined")


In [ ]:
# Training history
best_val_auc = 0.0
best_val_acc = 0.0
train_history = {'loss': [], 'acc': [], 'val_loss': [], 'val_acc': [], 'val_precision': [], 'val_recall': [], 'val_f1': [], 'val_auc': []}

print("\n" + "="*70)
print("🚀 STARTING STATE-OF-THE-ART TRAINING")
print("="*70)

for epoch in range(CONFIG['num_epochs']):
    print(f"\n{'='*70}")
    print(f"Epoch {epoch+1}/{CONFIG['num_epochs']}")
    print(f"{'='*70}")
    
    # Train (with scaler for mixed precision)
    train_loss, train_acc = train_epoch_advanced(model, train_loader, criterion, optimizer, CONFIG['device'], epoch, scaler=scaler)
    
    # Validate
    val_loss, val_acc, val_precision, val_recall, val_f1, val_auc = validate(model, valid_loader, criterion, CONFIG['device'])
    
    # Update learning rate
    scheduler.step()
    
    # Save history
    train_history['loss'].append(train_loss)
    train_history['acc'].append(train_acc)
    train_history['val_loss'].append(val_loss)
    train_history['val_acc'].append(val_acc)
    train_history['val_precision'].append(val_precision)
    train_history['val_recall'].append(val_recall)
    train_history['val_f1'].append(val_f1)
    train_history['val_auc'].append(val_auc)
    
    # Save best model (based on AUC) - BACKEND COMPATIBLE FORMAT
    if val_auc > best_val_auc:
        best_val_auc = val_auc
        best_val_acc = val_acc
        
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),  # Backend expects this key
            'optimizer_state_dict': optimizer.state_dict(),
            'val_auc': val_auc,
            'val_acc': val_acc,
            'val_precision': val_precision,
            'val_recall': val_recall,
            'val_f1': val_f1,
            'config': CONFIG,
            'model_name': CONFIG['model_name'],
            'history': train_history
        }, CONFIG['save_path'])
        print(f"✅ Saved best model (AUC: {val_auc:.4f}, Acc: {val_acc:.4f})")
    
    print(f"\nTrain - Loss: {train_loss:.4f}, Acc: {train_acc:.2f}%")
    print(f"Val   - Loss: {val_loss:.4f}, Acc: {val_acc:.4f}, AUC: {val_auc:.4f}")
    print(f"Val   - Precision: {val_precision:.4f}, Recall: {val_recall:.4f}, F1: {val_f1:.4f}")

print("\n✅ Training Complete!")


In [ ]:
print("\n" + "="*70)
print("📊 FINAL TEST SET EVALUATION")
print("="*70)

# Load best model
checkpoint = torch.load(CONFIG['save_path'], map_location=CONFIG['device'])
model.load_state_dict(checkpoint['model_state_dict'])
print("✅ Loaded best model from checkpoint")

# Evaluate on test set
test_loss, test_acc, test_precision, test_recall, test_f1, test_auc = validate(model, test_loader, criterion, CONFIG['device'])

print(f"\n🎯 Test Set Results:")
print(f"   Accuracy:  {test_acc:.4f} ({test_acc*100:.2f}%)")
print(f"   Precision: {test_precision:.4f}")
print(f"   Recall:    {test_recall:.4f}")
print(f"   F1-Score:  {test_f1:.4f}")
print(f"   AUC-ROC:   {test_auc:.4f}")

# Confusion Matrix
model.eval()
all_preds = []
all_labels = []
with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(CONFIG['device'])
        outputs = model(images)
        _, preds = outputs.max(1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())

cm = confusion_matrix(all_labels, all_preds)
print(f"\n📊 Confusion Matrix:")
print(f"   [[TN={cm[0,0]}, FP={cm[0,1]}],")
print(f"    [FN={cm[1,0]}, TP={cm[1,1]}]]")

# Save final model in BACKEND-COMPATIBLE FORMAT
final_checkpoint = {
    'model_state_dict': model.state_dict(),  # Backend expects this
    'config': CONFIG,
    'model_name': CONFIG['model_name'],
    'test_metrics': {
        'accuracy': test_acc,
        'precision': test_precision,
        'recall': test_recall,
        'f1': test_f1,
        'auc': test_auc
    },
    'val_auc': checkpoint.get('val_auc', test_auc),
    'val_acc': checkpoint.get('val_acc', test_acc),
}

torch.save(final_checkpoint, CONFIG['final_model_path'])

print(f"\n✅ Final model saved: {CONFIG['final_model_path']}")
print("\n📥 IMPORTANT: Download this file and place it in backend/model/")
print("   Then update backend/routes.py MODEL_PATH to point to this file")
print("   Update model creation code to use the same architecture (convnext_large)")
